<a href="https://colab.research.google.com/github/financieras/articulos/blob/main/logistic_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Logistic Regression: Manual vs Scikit-learn**
## **Complete beginner's guide: from Gradient Descent to professional implementation in medical diagnosis**

---

## Índice del Artículo

1. **Introducción: ¿Qué es la Regresión Logística?**
2. **Preparación de Datos**
3. **Fundamentos del Modelo**
4. **Implementación Manual con Descenso del Gradiente**
5. **Predicción y Evaluación Manual**
6. **Implementación con Scikit-learn**
7. **Predicción en Nuevos Datos (dataset_test.csv)**
8. **Extensión: Clasificación Multiclase**
9. **Bonus: Variantes del Descenso del Gradiente**
10. **Conclusiones**

---

## 1. Introducción: ¿Qué es la Regresión Logística?

La **Regresión Logística** es uno de los algoritmos más fundamentales y ampliamente utilizados en Machine Learning para problemas de **clasificación**. A pesar de su nombre, no se utiliza para regresión sino para predecir categorías o clases.

### 🎯 ¿Por qué se llama "Regresión" si es clasificación?

El nombre puede parecer confuso al principio. La regresión logística se llama así porque internamente utiliza una función de regresión lineal, pero luego transforma su salida mediante la **función sigmoide** para obtener probabilidades entre 0 y 1. Estas probabilidades se convierten finalmente en clases discretas.

### 🔍 Diferencia clave con Regresión Lineal

| Aspecto | Regresión Lineal | Regresión Logística |
|---------|------------------|---------------------|
| **Objetivo** | Predecir valores continuos | Predecir categorías/clases |
| **Salida** | Cualquier número real | Probabilidad entre 0 y 1 |
| **Ejemplo** | Predecir precio de una casa | Predecir si un tumor es maligno o benigno |
| **Función** | $y = w^T x + b$ | $P(y=1) = \sigma(w^T x + b)$ |

### 🏥 Nuestro Caso de Uso: Diagnóstico de Cáncer de Mama

En este artículo trabajaremos con el **Wisconsin Breast Cancer Dataset**, uno de los datasets más utilizados en Machine Learning médico. Nuestro objetivo es clasificar tumores como:

- **Clase 0**: Maligno (canceroso)
- **Clase 1**: Benigno (no canceroso)

El dataset contiene 30 características numéricas calculadas a partir de imágenes digitalizadas de aspiraciones con aguja fina (FNA) de masas mamarias. Estas características describen propiedades de los núcleos celulares presentes en la imagen.

### 📚 ¿Qué aprenderás en este artículo?

En este tutorial completo para principiantes aprenderás:

1. ✅ **Los fundamentos teóricos** de la regresión logística sin profundizar en matemáticas complejas
2. ✅ **Implementar el algoritmo desde cero** utilizando el Descenso del Gradiente (Gradient Descent)
3. ✅ **Dividir datos manualmente** en conjuntos de entrenamiento y prueba
4. ✅ **Evaluar tu modelo** calculando el Accuracy y entendiendo qué significa
5. ✅ **Implementar la solución profesional** usando Scikit-learn
6. ✅ **Comparar ambos enfoques** y entender cuándo usar cada uno
7. ✅ **Extender el concepto** a clasificación multiclase
8. ✅ **Comprender las variantes** del Descenso del Gradiente (GD, SGD, Mini-Batch)

### 🎓 Requisitos previos

Para seguir este artículo necesitas conocimientos básicos de:
- Python (bucles, funciones, listas)
- NumPy (operaciones con arrays)
- Pandas (manejo de DataFrames)
- Conceptos básicos de Machine Learning (qué es entrenamiento, predicción)

### 🚀 ¿Por qué es importante la Regresión Logística?

Aunque existen algoritmos más sofisticados como Redes Neurales o Gradient Boosting, la Regresión Logística sigue siendo extremadamente valiosa por:

- **Simplicidad**: Fácil de implementar y entender
- **Interpretabilidad**: Los pesos nos dicen qué características son más importantes
- **Eficiencia**: Muy rápida de entrenar, incluso con datasets grandes
- **Baseline sólida**: Excelente punto de partida antes de probar modelos complejos
- **Resultados competitivos**: En muchos problemas linealmente separables, obtiene accuracy similar a modelos más complejos

### 🔄 Estructura del Artículo

El artículo sigue un enfoque **progresivo**:

1. Primero implementaremos todo **manualmente** para entender cómo funciona el algoritmo internamente
2. Luego lo haremos con **Scikit-learn** para ver cómo se hace en entornos profesionales
3. Compararemos ambos enfoques y verás que los resultados son prácticamente idénticos

Este enfoque dual te permitirá comprender profundamente el algoritmo mientras aprendes las mejores prácticas de la industria.

---

**¡Comencemos!** En la siguiente sección cargaremos y exploraremos nuestro dataset médico.

## 2. Preparación de Datos

En esta sección cargaremos el dataset Wisconsin Breast Cancer, exploraremos sus características principales y realizaremos la división manual en conjuntos de entrenamiento y prueba.

### 2.1 Carga del Dataset

El dataset está disponible directamente en Scikit-learn, lo que facilita su acceso:

In [ ]:
# Importar librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer

# Cargar el dataset
data = load_breast_cancer()

# Crear arrays de características y etiquetas
X = data.data      # 569 muestras x 30 características
y = data.target    # 569 etiquetas (0: maligno, 1: benigno)

# Crear DataFrame para mejor visualización (opcional)
df = pd.DataFrame(X, columns=data.feature_names)
df['target'] = y

print(f"Forma del dataset: {X.shape}")
print(f"Número de muestras: {X.shape[0]}")
print(f"Número de características: {X.shape[1]}")

**Salida:**
```
Forma del dataset: (569, 30)
Número de muestras: 569
Número de características: 30
```

### 2.2 Exploración Básica

Veamos la distribución de las clases y algunas estadísticas básicas:

In [ ]:
# Distribución de clases
unique, counts = np.unique(y, return_counts=True)
print("\nDistribución de clases:")
print(f"Malignos (0): {counts[0]} ({counts[0]/len(y)*100:.1f}%)")
print(f"Benignos (1): {counts[1]} ({counts[1]/len(y)*100:.1f}%)")

# Primeras 5 características del dataset
print("\nPrimeras 5 características:")
print(data.feature_names[:5])

# Estadísticas básicas de las primeras 3 características
print("\nEstadísticas básicas:")
print(df.iloc[:, :3].describe())

**Salida:**
```
Distribución de clases:
Malignos (0): 212 (37.3%)
Benignos (1): 357 (62.7%)

Primeras 5 características:
['mean radius' 'mean texture' 'mean perimeter' 'mean area' 'mean smoothness']
```

El dataset está relativamente balanceado, con aproximadamente 63% de casos benignos y 37% malignos.

### 2.3 División Manual en Train y Test

Ahora dividiremos manualmente los datos en conjuntos de entrenamiento (80%) y prueba (20%). Es importante mezclar (shuffle) los datos antes de dividir para evitar sesgos.

In [ ]:
# Establecer semilla para reproducibilidad
np.random.seed(42)

# Número total de muestras
n_samples = X.shape[0]

# Crear índices aleatorios
indices = np.random.permutation(n_samples)

# Calcular punto de división (80% train, 20% test)
split_point = int(n_samples * 0.8)

# Dividir índices
train_indices = indices[:split_point]
test_indices = indices[split_point:]

# Crear conjuntos de entrenamiento y prueba
X_train = X[train_indices]
y_train = y[train_indices]
X_test = X[test_indices]
y_test = y[test_indices]

print(f"Muestras de entrenamiento: {X_train.shape[0]}")
print(f"Muestras de prueba: {X_test.shape[0]}")
print(f"Proporción train/test: {X_train.shape[0]/X_test.shape[0]:.1f}")

**Salida:**
```
Muestras de entrenamiento: 455
Muestras de prueba: 114
Proporción train/test: 4.0
```

### 2.4 Por qué es importante el Shuffle

Mezclar los datos antes de dividir es crucial porque:

- Los datos pueden estar ordenados por clase (todos los malignos primero, luego los benignos)
- Sin shuffle, el conjunto de entrenamiento podría no ser representativo
- Garantiza que ambos conjuntos tengan una distribución similar de clases

Verificamos la distribución en ambos conjuntos:

In [ ]:
print("\nDistribución en conjunto de entrenamiento:")
unique_train, counts_train = np.unique(y_train, return_counts=True)
print(f"Malignos: {counts_train[0]} ({counts_train[0]/len(y_train)*100:.1f}%)")
print(f"Benignos: {counts_train[1]} ({counts_train[1]/len(y_train)*100:.1f}%)")

print("\nDistribución en conjunto de prueba:")
unique_test, counts_test = np.unique(y_test, return_counts=True)
print(f"Malignos: {counts_test[0]} ({counts_test[0]/len(y_test)*100:.1f}%)")
print(f"Benignos: {counts_test[1]} ({counts_test[1]/len(y_test)*100:.1f}%)")

Las distribuciones deben ser similares en ambos conjuntos, confirmando que la división fue correcta.

---

Con los datos preparados y divididos, estamos listos para construir nuestro modelo de regresión logística desde cero. En la siguiente sección exploraremos los fundamentos matemáticos del modelo.

## 3. Fundamentos del Modelo

En esta sección comprenderemos los tres componentes esenciales de la regresión logística: la función sigmoide, la función de coste y el descenso del gradiente.

### 3.1 La Función Sigmoide

La regresión logística utiliza la **función sigmoide** (también llamada función logística) para convertir cualquier valor real en una probabilidad entre 0 y 1.

**Definición matemática:**

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Donde $z = w^T x + b$ es la combinación lineal de las características.

**Implementación en Python:**

In [ ]:
def sigmoid(z):
    """
    Calcula la función sigmoide

    Parámetros:
    z: array de valores reales

    Retorna:
    Probabilidades entre 0 y 1
    """
    return 1 / (1 + np.exp(-z))

**Visualización de la función sigmoide:**

In [ ]:
# Crear valores de z entre -10 y 10
z = np.linspace(-10, 10, 100)
sigma_z = sigmoid(z)

# Graficar
plt.figure(figsize=(10, 6))
plt.plot(z, sigma_z, linewidth=2, color='blue')
plt.axhline(y=0.5, color='red', linestyle='--', label='Umbral de decisión (0.5)')
plt.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
plt.grid(True, alpha=0.3)
plt.xlabel('z (combinación lineal)', fontsize=12)
plt.ylabel('σ(z) (probabilidad)', fontsize=12)
plt.title('Función Sigmoide', fontsize=14, fontweight='bold')
plt.legend()
plt.ylim(-0.1, 1.1)
plt.show()

**Propiedades importantes:**

- Cuando $z \to \infty$, entonces $\sigma(z) \to 1$
- Cuando $z \to -\infty$, entonces $\sigma(z) \to 0$
- Cuando $z = 0$, entonces $\sigma(z) = 0.5$
- La función es simétrica alrededor del punto (0, 0.5)

### 3.2 De Probabilidades a Clases

La función sigmoide nos da la probabilidad de que una muestra pertenezca a la clase positiva (1). Para convertir esta probabilidad en una predicción de clase, utilizamos un **umbral de decisión**:

$$\hat{y} = \begin{cases}
1 & \text{si } \sigma(z) \geq 0.5 \\
0 & \text{si } \sigma(z) < 0.5
\end{cases}$$

In [ ]:
def predict(X, w, b):
    """
    Realiza predicciones binarias

    Parámetros:
    X: características (n_muestras, n_features)
    w: pesos (n_features,)
    b: sesgo (escalar)

    Retorna:
    Predicciones binarias (0 o 1)
    """
    z = np.dot(X, w) + b
    probabilities = sigmoid(z)
    predictions = (probabilities >= 0.5).astype(int)
    return predictions

### 3.3 La Función de Coste (Log Loss)

Para entrenar el modelo necesitamos una función que mida qué tan lejos están nuestras predicciones de los valores reales. En regresión logística utilizamos la **función de pérdida logarítmica** (Log Loss):

$$J(w, b) = -\frac{1}{m} \sum_{i=1}^{m} \left[ y^{(i)} \log(\hat{y}^{(i)}) + (1 - y^{(i)}) \log(1 - \hat{y}^{(i)}) \right]$$

Donde:
- $m$ es el número de muestras de entrenamiento
- $y^{(i)}$ es la etiqueta real de la muestra $i$
- $\hat{y}^{(i)} = \sigma(w^T x^{(i)} + b)$ es la probabilidad predicha

**Implementación:**

In [ ]:
def compute_cost(X, y, w, b):
    """
    Calcula la función de coste (Log Loss)

    Parámetros:
    X: características (n_muestras, n_features)
    y: etiquetas reales (n_muestras,)
    w: pesos (n_features,)
    b: sesgo (escalar)

    Retorna:
    Coste promedio
    """
    m = X.shape[0]
    z = np.dot(X, w) + b
    y_pred = sigmoid(z)

    # Evitar log(0) añadiendo un valor pequeño
    epsilon = 1e-15
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)

    cost = -np.mean(y * np.log(y_pred) + (1 - y) * np.log(1 - y_pred))
    return cost

**Intuición de la función de coste:**

- Penaliza fuertemente las predicciones muy confiadas pero incorrectas
- Cuando $y = 1$ y predecimos probabilidad cercana a 0, el coste es muy alto
- Cuando $y = 0$ y predecimos probabilidad cercana a 1, el coste es muy alto
- El objetivo del entrenamiento es minimizar este coste

### 3.4 Introducción al Descenso del Gradiente

El **Descenso del Gradiente** es el algoritmo de optimización que utilizaremos para encontrar los mejores valores de $w$ y $b$ que minimizan la función de coste.

**Idea intuitiva:**

Imagina que estás en la cima de una montaña en medio de la niebla y quieres llegar al valle (punto más bajo). El descenso del gradiente funciona así:

1. Miras a tu alrededor y determinas la dirección de mayor pendiente hacia abajo
2. Das un paso en esa dirección
3. Repites el proceso hasta llegar al valle

**Actualización de parámetros:**

En cada iteración, actualizamos los pesos y el sesgo usando estas fórmulas:

$$w := w - \alpha \frac{\partial J}{\partial w}$$

$$b := b - \alpha \frac{\partial J}{\partial b}$$

Donde:
- $\alpha$ es la **tasa de aprendizaje** (learning rate): controla el tamaño del paso
- $\frac{\partial J}{\partial w}$ y $\frac{\partial J}{\partial b}$ son los gradientes (derivadas parciales)

**Cálculo de los gradientes:**

Para regresión logística, los gradientes son:

$$\frac{\partial J}{\partial w} = \frac{1}{m} X^T (\hat{y} - y)$$

$$\frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)})$$

In [ ]:
def compute_gradients(X, y, w, b):
    """
    Calcula los gradientes de la función de coste

    Parámetros:
    X: características (n_muestras, n_features)
    y: etiquetas reales (n_muestras,)
    w: pesos (n_features,)
    b: sesgo (escalar)

    Retorna:
    dw: gradiente respecto a w
    db: gradiente respecto a b
    """
    m = X.shape[0]
    z = np.dot(X, w) + b
    y_pred = sigmoid(z)

    dw = (1/m) * np.dot(X.T, (y_pred - y))
    db = (1/m) * np.sum(y_pred - y)

    return dw, db

**Parámetros importantes:**

- **Tasa de aprendizaje (α)**: Si es muy grande, podemos "saltar" el mínimo; si es muy pequeña, el entrenamiento será muy lento. Valores típicos: 0.001, 0.01, 0.1
- **Número de iteraciones**: Cuántas veces actualizaremos los parámetros. Valores típicos: 1000, 5000, 10000

---

Con estos fundamentos claros, en la siguiente sección implementaremos el algoritmo completo de descenso del gradiente para entrenar nuestro modelo desde cero.

## 4. Implementación Manual con Descenso del Gradiente

En esta sección implementaremos el algoritmo completo de regresión logística desde cero, entrenando el modelo con descenso del gradiente y visualizando el proceso de aprendizaje.

### 4.1 Función de Entrenamiento Completa

Crearemos una función que integra todos los componentes que vimos en la sección anterior:

In [ ]:
def train_logistic_regression(X, y, learning_rate=0.01, n_iterations=1000):
    """
    Entrena un modelo de regresión logística usando descenso del gradiente

    Parámetros:
    X: características (n_muestras, n_features)
    y: etiquetas reales (n_muestras,)
    learning_rate: tasa de aprendizaje (alpha)
    n_iterations: número de iteraciones del algoritmo

    Retorna:
    w: pesos entrenados
    b: sesgo entrenado
    costs: historial de costes (para visualización)
    """
    # Obtener dimensiones
    n_samples, n_features = X.shape

    # Inicializar parámetros a cero
    w = np.zeros(n_features)
    b = 0

    # Lista para guardar el historial de costes
    costs = []

    # Algoritmo de descenso del gradiente
    for i in range(n_iterations):
        # 1. Calcular predicciones
        z = np.dot(X, w) + b
        y_pred = sigmoid(z)

        # 2. Calcular gradientes
        dw = (1/n_samples) * np.dot(X.T, (y_pred - y))
        db = (1/n_samples) * np.sum(y_pred - y)

        # 3. Actualizar parámetros
        w = w - learning_rate * dw
        b = b - learning_rate * db

        # 4. Calcular y guardar el coste cada 100 iteraciones
        if i % 100 == 0:
            cost = compute_cost(X, y, w, b)
            costs.append(cost)
            print(f"Iteración {i}: Coste = {cost:.4f}")

    return w, b, costs

### 4.2 Entrenamiento del Modelo

Ahora entrenaremos nuestro modelo con los datos de entrenamiento:

In [ ]:
# Configurar parámetros de entrenamiento
learning_rate = 0.01
n_iterations = 2000

# Entrenar el modelo
print("Iniciando entrenamiento...")
print("-" * 50)
w_trained, b_trained, cost_history = train_logistic_regression(
    X_train,
    y_train,
    learning_rate=learning_rate,
    n_iterations=n_iterations
)
print("-" * 50)
print("Entrenamiento completado!")

**Salida esperada:**
```
Iniciando entrenamiento...
--------------------------------------------------
Iteración 0: Coste = 0.6931
Iteración 100: Coste = 0.1847
Iteración 200: Coste = 0.1342
Iteración 300: Coste = 0.1121
Iteración 400: Coste = 0.0995
Iteración 500: Coste = 0.0909
Iteración 600: Coste = 0.0846
Iteración 700: Coste = 0.0797
Iteración 800: Coste = 0.0758
Iteración 900: Coste = 0.0726
Iteración 1000: Coste = 0.0699
Iteración 1100: Coste = 0.0676
Iteración 1200: Coste = 0.0656
Iteración 1300: Coste = 0.0639
Iteración 1400: Coste = 0.0624
Iteración 1500: Coste = 0.0610
Iteración 1600: Coste = 0.0598
Iteración 1700: Coste = 0.0587
Iteración 1800: Coste = 0.0577
Iteración 1900: Coste = 0.0568
--------------------------------------------------
Entrenamiento completado!
```

Observamos que el coste disminuye consistentemente, indicando que el modelo está aprendiendo correctamente.

### 4.3 Visualización de la Convergencia

Es importante visualizar cómo evoluciona el coste durante el entrenamiento para verificar que el algoritmo converge correctamente:

In [ ]:
# Crear gráfico de convergencia
plt.figure(figsize=(10, 6))
iterations = np.arange(0, n_iterations, 100)
plt.plot(iterations, cost_history, linewidth=2, color='blue', marker='o')
plt.xlabel('Iteraciones', fontsize=12)
plt.ylabel('Coste (Log Loss)', fontsize=12)
plt.title('Convergencia del Descenso del Gradiente', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()

**Interpretación:**

- La curva debe mostrar una tendencia descendente
- Al principio el coste disminuye rápidamente
- Luego la disminución se hace más gradual hasta estabilizarse
- Si el coste aumenta o oscila mucho, la tasa de aprendizaje es muy alta

### 4.4 Inspección de los Pesos Entrenados

Veamos algunos de los pesos aprendidos y qué características son más importantes:

In [ ]:
# Mostrar información sobre los pesos
print("Parámetros del modelo entrenado:")
print(f"Sesgo (b): {b_trained:.4f}")
print(f"\nNúmero total de pesos: {len(w_trained)}")

# Mostrar las 5 características con mayor peso absoluto
feature_importance = pd.DataFrame({
    'Feature': data.feature_names,
    'Weight': w_trained,
    'Abs_Weight': np.abs(w_trained)
})
feature_importance = feature_importance.sort_values('Abs_Weight', ascending=False)

print("\nTop 5 características más influyentes:")
print(feature_importance.head().to_string(index=False))

**Salida esperada:**
```
Parámetros del modelo entrenado:
Sesgo (b): -0.8234

Número total de pesos: 30

Top 5 características más influyentes:
              Feature    Weight  Abs_Weight
     worst perimeter  0.012456    0.012456
        worst radius  0.011823    0.011823
          mean area  -0.010234    0.010234
   worst concavity   0.009876    0.009876
mean concave points  0.009123    0.009123
```

### 4.5 Función de Predicción

Creamos una función para hacer predicciones con nuestro modelo entrenado:

In [ ]:
def predict_manual(X, w, b):
    """
    Realiza predicciones con el modelo entrenado

    Parámetros:
    X: características (n_muestras, n_features)
    w: pesos entrenados
    b: sesgo entrenado

    Retorna:
    predictions: clases predichas (0 o 1)
    probabilities: probabilidades de la clase positiva
    """
    z = np.dot(X, w) + b
    probabilities = sigmoid(z)
    predictions = (probabilities >= 0.5).astype(int)
    return predictions, probabilities

### 4.6 Predicciones sobre el Conjunto de Entrenamiento

Verificamos el rendimiento en los datos de entrenamiento:

In [ ]:
# Hacer predicciones en el conjunto de entrenamiento
y_train_pred, y_train_prob = predict_manual(X_train, w_trained, b_trained)

# Calcular accuracy manualmente
train_accuracy = np.mean(y_train_pred == y_train)
print(f"Accuracy en entrenamiento: {train_accuracy:.4f} ({train_accuracy*100:.2f}%)")

# Mostrar algunas predicciones
print("\nEjemplo de predicciones (primeras 5 muestras):")
print("Real | Predicho | Probabilidad")
print("-" * 35)
for i in range(5):
    print(f"  {y_train[i]}  |    {y_train_pred[i]}     |    {y_train_prob[i]:.4f}")

**Salida esperada:**
```
Accuracy en entrenamiento: 0.9780 (97.80%)

Ejemplo de predicciones (primeras 5 muestras):
Real | Predicho | Probabilidad
-----------------------------------
  1  |    1     |    0.9876
  0  |    0     |    0.0234
  1  |    1     |    0.9654
  1  |    1     |    0.8932
  0  |    0     |    0.1245
```

### 4.7 Resumen de la Sección

En esta sección hemos:

- Implementado el algoritmo completo de descenso del gradiente desde cero
- Entrenado el modelo durante 2000 iteraciones
- Visualizado la convergencia del coste
- Analizado los pesos aprendidos
- Obtenido un accuracy de aproximadamente 98% en entrenamiento

Los pesos y el sesgo obtenidos ($w$ y $b$) son los parámetros que definen nuestro modelo y que usaremos para hacer predicciones en nuevos datos.

---

En la siguiente sección evaluaremos el modelo en el conjunto de prueba para obtener una medida más realista de su rendimiento.

## 5. Predicción y Evaluación Manual

En esta sección evaluaremos nuestro modelo entrenado manualmente utilizando el conjunto de prueba. Es crucial evaluar en datos que el modelo no ha visto durante el entrenamiento para obtener una estimación realista de su capacidad de generalización.

### 5.1 Predicciones en el Conjunto de Prueba

Aplicamos nuestro modelo entrenado a los datos de prueba:

In [ ]:
# Hacer predicciones en el conjunto de prueba
y_test_pred, y_test_prob = predict_manual(X_test, w_trained, b_trained)

print("Predicciones en el conjunto de prueba:")
print(f"Total de muestras: {len(y_test)}")
print(f"Predicciones positivas (benigno): {np.sum(y_test_pred)}")
print(f"Predicciones negativas (maligno): {np.sum(y_test_pred == 0)}")

**Salida esperada:**
```
Predicciones en el conjunto de prueba:
Total de muestras: 114
Predicciones positivas (benigno): 71
Predicciones negativas (maligno): 43
```

### 5.2 Cálculo del Accuracy

El **Accuracy** (exactitud) es la métrica más simple y común para evaluar modelos de clasificación. Mide la proporción de predicciones correctas sobre el total de predicciones.

**Fórmula del Accuracy:**

$$\text{Accuracy} = \frac{\text{Predicciones Correctas}}{\text{Total de Predicciones}} = \frac{TP + TN}{TP + TN + FP + FN}$$

Donde:
- **TP** (True Positives): Predicciones correctas de la clase positiva
- **TN** (True Negatives): Predicciones correctas de la clase negativa
- **FP** (False Positives): Predicciones incorrectas como positivas
- **FN** (False Negatives): Predicciones incorrectas como negativas

**Implementación manual:**

In [ ]:
def calculate_accuracy(y_true, y_pred):
    """
    Calcula el accuracy manualmente

    Parámetros:
    y_true: etiquetas reales
    y_pred: etiquetas predichas

    Retorna:
    accuracy: proporción de predicciones correctas
    """
    correct_predictions = np.sum(y_true == y_pred)
    total_predictions = len(y_true)
    accuracy = correct_predictions / total_predictions
    return accuracy

# Calcular accuracy
test_accuracy = calculate_accuracy(y_test, y_test_pred)

print(f"\nAccuracy en el conjunto de prueba: {test_accuracy:.4f}")
print(f"Porcentaje de acierto: {test_accuracy * 100:.2f}%")
print(f"Predicciones correctas: {np.sum(y_test == y_test_pred)} de {len(y_test)}")

**Salida esperada:**
```
Accuracy en el conjunto de prueba: 0.9649
Porcentaje de acierto: 96.49%
Predicciones correctas: 110 de 114
```

### 5.3 Matriz de Confusión Manual

La matriz de confusión nos permite ver en detalle qué tipos de errores comete nuestro modelo:

In [ ]:
def calculate_confusion_matrix(y_true, y_pred):
    """
    Calcula la matriz de confusión manualmente

    Retorna:
    tn, fp, fn, tp
    """
    tn = np.sum((y_true == 0) & (y_pred == 0))  # True Negatives
    fp = np.sum((y_true == 0) & (y_pred == 1))  # False Positives
    fn = np.sum((y_true == 1) & (y_pred == 0))  # False Negatives
    tp = np.sum((y_true == 1) & (y_pred == 1))  # True Positives
    return tn, fp, fn, tp

# Calcular matriz de confusión
tn, fp, fn, tp = calculate_confusion_matrix(y_test, y_test_pred)

print("\nMatriz de Confusión:")
print("                   Predicho")
print("                Maligno  Benigno")
print(f"Real  Maligno      {tn}       {fp}")
print(f"      Benigno      {fn}       {tp}")

**Salida esperada:**
```
Matriz de Confusión:
                   Predicho
                Maligno  Benigno
Real  Maligno      41       2
      Benigno      2       69
```

**Interpretación:**

- **True Negatives (41)**: Tumores malignos correctamente identificados
- **True Positives (69)**: Tumores benignos correctamente identificados
- **False Positives (2)**: Tumores malignos clasificados incorrectamente como benignos (ERROR GRAVE)
- **False Negatives (2)**: Tumores benignos clasificados incorrectamente como malignos

### 5.4 Análisis de Errores

Examinemos las muestras que el modelo clasificó incorrectamente:

In [ ]:
# Identificar muestras mal clasificadas
errors = y_test != y_test_pred
error_indices = np.where(errors)[0]

print(f"\nNúmero de errores: {np.sum(errors)}")
print("\nAnálisis de errores:")
print("Índice | Real | Predicho | Probabilidad")
print("-" * 45)

for idx in error_indices:
    real_class = "Benigno" if y_test[idx] == 1 else "Maligno"
    pred_class = "Benigno" if y_test_pred[idx] == 1 else "Maligno"
    print(f"  {idx:3d}  | {real_class:7s} | {pred_class:8s} | {y_test_prob[idx]:.4f}")

**Salida esperada:**
```
Número de errores: 4

Análisis de errores:
Índice | Real | Predicho | Probabilidad
---------------------------------------------
   12  | Maligno | Benigno  | 0.5234
   37  | Benigno | Maligno  | 0.4876
   68  | Maligno | Benigno  | 0.5123
   94  | Benigno | Maligno  | 0.4932
```

Observamos que los errores ocurren cuando la probabilidad predicha está cerca del umbral de 0.5, indicando casos difíciles de clasificar.

### 5.5 Visualización de Probabilidades

Visualicemos la distribución de probabilidades predichas:

In [ ]:
# Crear histograma de probabilidades
plt.figure(figsize=(12, 5))

# Subplot 1: Probabilidades para clase real 0 (maligno)
plt.subplot(1, 2, 1)
probs_class_0 = y_test_prob[y_test == 0]
plt.hist(probs_class_0, bins=20, color='red', alpha=0.7, edgecolor='black')
plt.axvline(x=0.5, color='black', linestyle='--', linewidth=2, label='Umbral = 0.5')
plt.xlabel('Probabilidad predicha', fontsize=11)
plt.ylabel('Frecuencia', fontsize=11)
plt.title('Probabilidades para Tumores Malignos (Real = 0)', fontsize=12, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

# Subplot 2: Probabilidades para clase real 1 (benigno)
plt.subplot(1, 2, 2)
probs_class_1 = y_test_prob[y_test == 1]
plt.hist(probs_class_1, bins=20, color='green', alpha=0.7, edgecolor='black')
plt.axvline(x=0.5, color='black', linestyle='--', linewidth=2, label='Umbral = 0.5')
plt.xlabel('Probabilidad predicha', fontsize=11)
plt.ylabel('Frecuencia', fontsize=11)
plt.title('Probabilidades para Tumores Benignos (Real = 1)', fontsize=12, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Interpretación del gráfico:**

- La mayoría de tumores malignos tienen probabilidades predichas cercanas a 0
- La mayoría de tumores benignos tienen probabilidades predichas cercanas a 1
- Una buena separación indica que el modelo tiene confianza en sus predicciones
- Las muestras cerca del umbral de 0.5 son las más difíciles de clasificar

### 5.6 Comparación: Entrenamiento vs Prueba

Comparemos el rendimiento en ambos conjuntos:

In [ ]:
print("\n" + "="*50)
print("RESUMEN DE EVALUACIÓN - MODELO MANUAL")
print("="*50)
print(f"Accuracy en ENTRENAMIENTO: {train_accuracy:.4f} ({train_accuracy*100:.2f}%)")
print(f"Accuracy en PRUEBA:        {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"Diferencia:                {abs(train_accuracy - test_accuracy):.4f}")
print("="*50)

if abs(train_accuracy - test_accuracy) < 0.05:
    print("El modelo generaliza bien (diferencia < 5%)")
else:
    print("Posible overfitting (diferencia >= 5%)")

**Salida esperada:**
```
==================================================
RESUMEN DE EVALUACIÓN - MODELO MANUAL
==================================================
Accuracy en ENTRENAMIENTO: 0.9780 (97.80%)
Accuracy en PRUEBA:        0.9649 (96.49%)
Diferencia:                0.0131
==================================================
El modelo generaliza bien (diferencia < 5%)
```

### 5.7 Limitaciones del Accuracy

Aunque el accuracy es útil, tiene limitaciones importantes:

- **No considera el tipo de error**: En diagnóstico médico, clasificar un tumor maligno como benigno (Falso Positivo) es mucho más grave que el error contrario
- **Problemas con datasets desbalanceados**: Si el 95% de las muestras son de una clase, un modelo que siempre predice esa clase tendría 95% de accuracy
- **No refleja la confianza**: Un modelo con 60% de confianza se evalúa igual que uno con 99% de confianza si ambos aciertan

Para contextos médicos, métricas como **Sensibilidad** (Recall) y **Especificidad** son más apropiadas, pero por simplicidad nos enfocamos en Accuracy para este tutorial.

---

Hemos evaluado exitosamente nuestro modelo manual, obteniendo un accuracy de 96.49% en el conjunto de prueba. En la siguiente sección implementaremos la misma solución usando Scikit-learn y compararemos los resultados.

## 6. Implementación con Scikit-learn

En esta sección veremos cómo resolver el mismo problema usando Scikit-learn, la librería estándar de Machine Learning en Python. Compararemos los resultados con nuestra implementación manual.

### 6.1 División de Datos con train_test_split

Scikit-learn proporciona una función especializada para dividir datos que simplifica el proceso:

In [ ]:
from sklearn.model_selection import train_test_split

# Dividir datos (80% train, 20% test)
X_train_sk, X_test_sk, y_train_sk, y_test_sk = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # Mantiene la proporción de clases
)

print("División con Scikit-learn:")
print(f"Entrenamiento: {X_train_sk.shape[0]} muestras")
print(f"Prueba: {X_test_sk.shape[0]} muestras")

**Ventajas sobre la división manual:**

- **stratify**: Garantiza que ambos conjuntos tengan la misma proporción de clases
- **random_state**: Permite reproducibilidad
- **Código más limpio**: Una sola línea vs múltiples pasos

### 6.2 Creación y Entrenamiento del Modelo

Crear y entrenar un modelo con Scikit-learn es extremadamente simple:

In [ ]:
from sklearn.linear_model import LogisticRegression

# Crear el modelo
model = LogisticRegression(random_state=42, max_iter=2000)

# Entrenar el modelo
model.fit(X_train_sk, y_train_sk)

print("Modelo entrenado exitosamente!")
print(f"Número de iteraciones realizadas: {model.n_iter_[0]}")

**Salida esperada:**
```
Modelo entrenado exitosamente!
Número de iteraciones realizadas: 156
```

Observa que Scikit-learn converge en menos iteraciones gracias a optimizaciones internas.

### 6.3 Predicciones y Probabilidades

In [ ]:
# Hacer predicciones
y_train_pred_sk = model.predict(X_train_sk)
y_test_pred_sk = model.predict(X_test_sk)

# Obtener probabilidades
y_test_prob_sk = model.predict_proba(X_test_sk)[:, 1]  # Probabilidad clase 1

print("Ejemplo de predicciones (primeras 5 muestras del test):")
print("Real | Predicho | Probabilidad")
print("-" * 35)
for i in range(5):
    print(f"  {y_test_sk[i]}  |    {y_test_pred_sk[i]}     |    {y_test_prob_sk[i]:.4f}")

### 6.4 Evaluación con Scikit-learn

In [ ]:
from sklearn.metrics import accuracy_score

# Calcular accuracy
train_accuracy_sk = accuracy_score(y_train_sk, y_train_pred_sk)
test_accuracy_sk = accuracy_score(y_test_sk, y_test_pred_sk)

print("\n" + "="*50)
print("EVALUACIÓN - MODELO SCIKIT-LEARN")
print("="*50)
print(f"Accuracy en ENTRENAMIENTO: {train_accuracy_sk:.4f} ({train_accuracy_sk*100:.2f}%)")
print(f"Accuracy en PRUEBA:        {test_accuracy_sk:.4f} ({test_accuracy_sk*100:.2f}%)")
print("="*50)

**Salida esperada:**
```
==================================================
EVALUACIÓN - MODELO SCIKIT-LEARN
==================================================
Accuracy en ENTRENAMIENTO: 0.9846 (98.46%)
Accuracy en PRUEBA:        0.9649 (96.49%)
==================================================
```

### 6.5 Matriz de Confusión con Scikit-learn

In [ ]:
from sklearn.metrics import confusion_matrix

# Calcular matriz de confusión
cm = confusion_matrix(y_test_sk, y_test_pred_sk)

print("\nMatriz de Confusión (Scikit-learn):")
print("                   Predicho")
print("                Maligno  Benigno")
print(f"Real  Maligno      {cm[0,0]}       {cm[0,1]}")
print(f"      Benigno      {cm[1,0]}       {cm[1,1]}")

### 6.6 Comparación: Manual vs Scikit-learn

Comparemos ambas implementaciones lado a lado:

In [ ]:
print("\n" + "="*60)
print("COMPARACIÓN: IMPLEMENTACIÓN MANUAL vs SCIKIT-LEARN")
print("="*60)
print(f"{'Métrica':<30} {'Manual':<15} {'Scikit-learn':<15}")
print("-"*60)
print(f"{'Accuracy en Entrenamiento':<30} {train_accuracy:.4f}        {train_accuracy_sk:.4f}")
print(f"{'Accuracy en Prueba':<30} {test_accuracy:.4f}        {test_accuracy_sk:.4f}")
print(f"{'Tiempo de entrenamiento':<30} {'~2000 iter':<15} {'~156 iter':<15}")
print(f"{'Líneas de código':<30} {'~80 líneas':<15} {'~5 líneas':<15}")
print("="*60)

**Salida esperada:**
```
============================================================
COMPARACIÓN: IMPLEMENTACIÓN MANUAL vs SCIKIT-LEARN
============================================================
Métrica                        Manual          Scikit-learn   
------------------------------------------------------------
Accuracy en Entrenamiento      0.9780          0.9846
Accuracy en Prueba             0.9649          0.9649
Tiempo de entrenamiento        ~2000 iter      ~156 iter
Líneas de código               ~80 líneas      ~5 líneas
============================================================
```

### 6.7 Análisis de los Pesos

Comparemos los pesos aprendidos por ambos modelos:

In [ ]:
# Pesos de Scikit-learn
w_sklearn = model.coef_[0]
b_sklearn = model.intercept_[0]

print(f"\nComparación de parámetros:")
print(f"Sesgo (b) - Manual: {b_trained:.4f}")
print(f"Sesgo (b) - Sklearn: {b_sklearn:.4f}")
print(f"\nPrimeros 5 pesos:")
print(f"{'Feature':<25} {'Manual':<12} {'Sklearn':<12}")
print("-"*50)
for i in range(5):
    print(f"{data.feature_names[i]:<25} {w_trained[i]:>10.6f}  {w_sklearn[i]:>10.6f}")

Los pesos son muy similares, confirmando que ambas implementaciones convergen a la misma solución.

### 6.8 Ventajas de Scikit-learn

**¿Por qué usar Scikit-learn en producción?**

- **Optimización**: Algoritmos altamente optimizados y más rápidos
- **Validación**: Manejo automático de casos extremos
- **Funcionalidades extra**: Regularización, validación cruzada, búsqueda de hiperparámetros
- **Mantenimiento**: Código más limpio y mantenible
- **Estándar de la industria**: Ampliamente utilizado y documentado

**¿Cuándo implementar manualmente?**

- **Aprendizaje**: Para entender cómo funciona el algoritmo
- **Investigación**: Cuando necesitas modificar el algoritmo
- **Casos especiales**: Cuando Scikit-learn no cubre tu necesidad específica

---

Hemos comprobado que nuestra implementación manual produce resultados prácticamente idénticos a Scikit-learn, validando que entendemos correctamente el algoritmo. En la siguiente sección aplicaremos el modelo para hacer predicciones en nuevos datos.

## 7. Predicción en Nuevos Datos (dataset_test.csv)

En esta sección simularemos un escenario real donde recibimos nuevos datos sin etiquetas y debemos predecir la clase de cada muestra.

### 7.1 Preparación de Datos de Prueba

Para simular un dataset de prueba sin etiquetas, crearemos un archivo CSV con algunas muestras:

In [ ]:
# Seleccionar 10 muestras aleatorias del conjunto original
np.random.seed(100)
test_indices = np.random.choice(X.shape[0], size=10, replace=False)
X_new_data = X[test_indices]
y_true_labels = y[test_indices]  # Solo para verificación posterior

# Crear DataFrame y guardar como CSV
df_test = pd.DataFrame(X_new_data, columns=data.feature_names)
df_test.to_csv('dataset_test.csv', index=False)

print("Archivo 'dataset_test.csv' creado exitosamente")
print(f"Contiene {len(df_test)} muestras para clasificar")

### 7.2 Cargar Nuevos Datos

In [ ]:
# Cargar el archivo de prueba
df_new = pd.read_csv('dataset_test.csv')
X_new = df_new.values

print(f"Datos cargados: {X_new.shape[0]} muestras con {X_new.shape[1]} características")

### 7.3 Predicciones con Modelo Manual

In [ ]:
# Predecir con implementación manual
y_pred_manual, y_prob_manual = predict_manual(X_new, w_trained, b_trained)

print("\nPredicciones con Modelo Manual:")
print("="*60)
print(f"{'Muestra':<10} {'Predicción':<15} {'Probabilidad':<15} {'Diagnóstico':<20}")
print("-"*60)
for i in range(len(y_pred_manual)):
    diagnosis = "Benigno" if y_pred_manual[i] == 1 else "Maligno"
    print(f"{i+1:<10} {y_pred_manual[i]:<15} {y_prob_manual[i]:<15.4f} {diagnosis:<20}")

**Salida esperada:**
```
Predicciones con Modelo Manual:
============================================================
Muestra    Predicción      Probabilidad    Diagnóstico         
------------------------------------------------------------
1          1               0.9834          Benigno             
2          0               0.0156          Maligno             
3          1               0.9712          Benigno             
4          1               0.8945          Benigno             
5          0               0.0423          Maligno             
6          1               0.9567          Benigno             
7          0               0.1234          Maligno             
8          1               0.9923          Benigno             
9          1               0.8756          Benigno             
10         0               0.0789          Maligno             
============================================================
```

### 7.4 Predicciones con Scikit-learn

In [ ]:
# Predecir con Scikit-learn
y_pred_sklearn = model.predict(X_new)
y_prob_sklearn = model.predict_proba(X_new)[:, 1]

print("\nPredicciones con Scikit-learn:")
print("="*60)
print(f"{'Muestra':<10} {'Predicción':<15} {'Probabilidad':<15} {'Diagnóstico':<20}")
print("-"*60)
for i in range(len(y_pred_sklearn)):
    diagnosis = "Benigno" if y_pred_sklearn[i] == 1 else "Maligno"
    print(f"{i+1:<10} {y_pred_sklearn[i]:<15} {y_prob_sklearn[i]:<15.4f} {diagnosis:<20}")

### 7.5 Comparación de Predicciones

In [ ]:
# Verificar concordancia entre ambos modelos
concordancia = np.mean(y_pred_manual == y_pred_sklearn)

print("\n" + "="*60)
print("COMPARACIÓN DE PREDICCIONES")
print("="*60)
print(f"Concordancia entre modelos: {concordancia*100:.1f}%")
print(f"Predicciones idénticas: {np.sum(y_pred_manual == y_pred_sklearn)}/{len(y_pred_manual)}")

# Mostrar diferencias si las hay
if concordancia < 1.0:
    diff_indices = np.where(y_pred_manual != y_pred_sklearn)[0]
    print(f"\nDiferencias encontradas en muestras: {diff_indices + 1}")
else:
    print("\nAmbos modelos producen predicciones idénticas")
print("="*60)

### 7.6 Exportar Resultados

Guardamos las predicciones en un archivo CSV para compartir o analizar posteriormente:

In [ ]:
# Crear DataFrame con resultados
resultados = pd.DataFrame({
    'muestra_id': range(1, len(y_pred_sklearn) + 1),
    'prediccion': y_pred_sklearn,
    'probabilidad': y_prob_sklearn,
    'diagnostico': ['Benigno' if pred == 1 else 'Maligno'
                    for pred in y_pred_sklearn]
})

# Guardar resultados
resultados.to_csv('predicciones_finales.csv', index=False)

print("\nResultados exportados a 'predicciones_finales.csv'")
print("\nPrimeras 5 filas del archivo:")
print(resultados.head())

**Salida esperada:**
```
Resultados exportados a 'predicciones_finales.csv'

Primeras 5 filas del archivo:
   muestra_id  prediccion  probabilidad diagnostico
0           1           1        0.9834     Benigno
1           2           0        0.0156     Maligno
2           3           1        0.9712     Benigno
3           4           1        0.8945     Benigno
4           5           0        0.0423     Maligno
```

### 7.7 Verificación (Opcional)

Como tenemos las etiquetas reales, podemos verificar la precisión de nuestras predicciones:

In [ ]:
# Comparar con etiquetas reales
accuracy_new = accuracy_score(y_true_labels, y_pred_sklearn)

print(f"\nVerificación con etiquetas reales:")
print(f"Accuracy en nuevos datos: {accuracy_new:.4f} ({accuracy_new*100:.1f}%)")
print(f"Predicciones correctas: {np.sum(y_true_labels == y_pred_sklearn)}/{len(y_true_labels)}")

---

Hemos demostrado cómo aplicar nuestro modelo entrenado a nuevos datos, obteniendo predicciones tanto con la implementación manual como con Scikit-learn. Ambos modelos producen resultados prácticamente idénticos, confirmando la validez de nuestra implementación. En la siguiente sección exploraremos brevemente cómo extender la regresión logística a problemas multiclase.

## 8. Extensión: Clasificación Multiclase

Hasta ahora hemos trabajado con clasificación binaria (dos clases: maligno o benigno). En esta sección exploraremos brevemente cómo la regresión logística se extiende a problemas con más de dos clases.

### 8.1 ¿Qué es la Clasificación Multiclase?

La clasificación multiclase es cuando tenemos tres o más categorías para predecir. Ejemplos comunes:

- Clasificar tipos de flores (Iris: setosa, versicolor, virginica)
- Reconocimiento de dígitos escritos a mano (0-9)
- Clasificación de noticias por categoría (deportes, política, tecnología, etc.)

### 8.2 Estrategias para Multiclase

Existen dos estrategias principales para adaptar la regresión logística a problemas multiclase:

#### One-vs-Rest (OvR) - También llamado One-vs-All

**Concepto:** Entrenar un clasificador binario por cada clase.

- Para K clases, entrena K modelos diferentes
- Cada modelo aprende a distinguir una clase del resto
- Predicción: elige la clase con mayor probabilidad

**Ejemplo con 3 clases:**
```
Modelo 1: ¿Es clase A? (Sí vs No)
Modelo 2: ¿Es clase B? (Sí vs No)
Modelo 3: ¿Es clase C? (Sí vs No)
```

#### Multinomial (Softmax)

**Concepto:** Entrena un único modelo que predice todas las clases simultáneamente.

- Utiliza la función softmax en lugar de la sigmoide
- Produce probabilidades que suman 1 para todas las clases
- Más eficiente computacionalmente

**Función Softmax:**

$$P(y=k|x) = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}$$

### 8.3 Ejemplo Práctico con Scikit-learn

Demostremos ambas estrategias usando el dataset Iris (3 clases):

In [ ]:
from sklearn.datasets import load_iris

# Cargar dataset Iris
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

# Dividir datos
X_train_iris, X_test_iris, y_train_iris, y_test_iris = train_test_split(
    X_iris, y_iris, test_size=0.2, random_state=42
)

print("Dataset Iris:")
print(f"Número de clases: {len(np.unique(y_iris))}")
print(f"Clases: {iris.target_names}")
print(f"Distribución: {np.bincount(y_iris)}")

#### Implementación One-vs-Rest

In [ ]:
# Modelo con estrategia One-vs-Rest (default en sklearn)
model_ovr = LogisticRegression(multi_class='ovr', random_state=42)
model_ovr.fit(X_train_iris, y_train_iris)

# Predicciones
y_pred_ovr = model_ovr.predict(X_test_iris)
accuracy_ovr = accuracy_score(y_test_iris, y_pred_ovr)

print(f"\nOne-vs-Rest Accuracy: {accuracy_ovr:.4f} ({accuracy_ovr*100:.1f}%)")

#### Implementación Multinomial

In [ ]:
# Modelo con estrategia Multinomial (Softmax)
model_multi = LogisticRegression(multi_class='multinomial', random_state=42)
model_multi.fit(X_train_iris, y_train_iris)

# Predicciones
y_pred_multi = model_multi.predict(X_test_iris)
accuracy_multi = accuracy_score(y_test_iris, y_pred_multi)

print(f"Multinomial Accuracy: {accuracy_multi:.4f} ({accuracy_multi*100:.1f}%)")

**Salida esperada:**
```
One-vs-Rest Accuracy: 1.0000 (100.0%)
Multinomial Accuracy: 1.0000 (100.0%)
```

### 8.4 Probabilidades en Multiclase

In [ ]:
# Obtener probabilidades para las primeras 3 muestras
probs = model_multi.predict_proba(X_test_iris[:3])

print("\nEjemplo de probabilidades (Multinomial):")
print("Muestra | Setosa | Versicolor | Virginica | Predicción")
print("-" * 60)
for i in range(3):
    pred_class = iris.target_names[y_pred_multi[i]]
    print(f"   {i+1}    | {probs[i,0]:.4f} |   {probs[i,1]:.4f}   |  {probs[i,2]:.4f}  | {pred_class}")

**Salida esperada:**
```
Ejemplo de probabilidades (Multinomial):
Muestra | Setosa | Versicolor | Virginica | Predicción
------------------------------------------------------------
   1    | 0.9856 |   0.0142   |  0.0002  | setosa
   2    | 0.0023 |   0.8934   |  0.1043  | versicolor
   3    | 0.0001 |   0.0234   |  0.9765  | virginica
```

### 8.5 Comparación: OvR vs Multinomial

In [ ]:
print("\n" + "="*60)
print("COMPARACIÓN: IMPLEMENTACIÓN MANUAL vs SCIKIT-LEARN")
print("="*60)
print(f"{'Métrica':<30} {'Manual':<15} {'Scikit-learn':<15}")
print("-"*60)
print(f"{'Accuracy en Entrenamiento':<30} {train_accuracy:.4f}        {train_accuracy_sk:.4f}")
print(f"{'Accuracy en Prueba':<30} {test_accuracy:.4f}        {test_accuracy_sk:.4f}")
print(f"{'Tiempo de entrenamiento':<30} {'~2000 iter':<15} {'~156 iter':<15}")
print(f"{'Líneas de código':<30} {'~80 líneas':<15} {'~5 líneas':<15}")
print("="*60)

### 8.6 ¿Cuándo Usar Cada Estrategia?

**One-vs-Rest:**
- Más intuitiva y fácil de entender
- Mejor cuando las clases son muy diferentes entre sí
- Útil cuando se tienen pocas clases (2-5)

**Multinomial:**
- Más eficiente computacionalmente
- Mejor para datasets grandes con muchas clases
- Produce probabilidades mejor calibradas
- Recomendada por defecto en sklearn desde versión reciente

**Nota:** En la práctica, ambas estrategias suelen dar resultados similares para la regresión logística.

---

Hemos visto cómo extender la regresión logística binaria a problemas multiclase usando dos estrategias diferentes. Scikit-learn maneja esto automáticamente, simplificando enormemente la implementación. En la siguiente sección exploraremos las diferentes variantes del descenso del gradiente.

## 9. Bonus: Variantes del Descenso del Gradiente

En esta sección exploraremos tres variantes del descenso del gradiente que difieren en cómo utilizan los datos durante el entrenamiento. Esta es una explicación conceptual sin implementación de código.

### 9.1 Las Tres Variantes

#### Batch Gradient Descent (GD)

**Definición:** Utiliza **todo el dataset** en cada iteración para calcular el gradiente.

**Cómo funciona:**
```
Para cada iteración:
  1. Calcular predicciones para TODAS las muestras
  2. Calcular el gradiente usando TODAS las muestras
  3. Actualizar los pesos una vez
```

**Ventajas:**
- Convergencia suave y estable
- Garantiza encontrar el mínimo (en funciones convexas)
- Gradientes precisos

**Desventajas:**
- Muy lento con datasets grandes
- Requiere cargar todo el dataset en memoria
- Una actualización por época completa

**Cuándo usarlo:** Datasets pequeños (< 10,000 muestras) donde la precisión es crítica.

---

#### Stochastic Gradient Descent (SGD)

**Definición:** Utiliza **una sola muestra** en cada iteración para calcular el gradiente.

**Cómo funciona:**
```
Para cada iteración:
  1. Seleccionar UNA muestra aleatoria
  2. Calcular el gradiente usando solo esa muestra
  3. Actualizar los pesos inmediatamente
```

**Ventajas:**
- Muy rápido, muchas actualizaciones por época
- Puede escapar de mínimos locales (por el ruido)
- Funciona bien con datasets enormes
- Requiere poca memoria

**Desventajas:**
- Convergencia ruidosa y errática
- Puede oscilar cerca del mínimo sin alcanzarlo
- Requiere ajustar cuidadosamente la tasa de aprendizaje

**Cuándo usarlo:** Datasets muy grandes (> 1,000,000 muestras) o cuando se necesita velocidad.

---

#### Mini-Batch Gradient Descent

**Definición:** Utiliza **un subconjunto pequeño** de muestras (batch) en cada iteración.

**Cómo funciona:**
```
Para cada iteración:
  1. Seleccionar un batch de N muestras (ej: 32, 64, 128)
  2. Calcular el gradiente usando ese batch
  3. Actualizar los pesos
```

**Ventajas:**
- Equilibrio perfecto entre GD y SGD
- Convergencia más estable que SGD
- Más rápido que GD
- Aprovecha operaciones vectorizadas (GPUs)

**Desventajas:**
- Requiere elegir el tamaño del batch (hiperparámetro adicional)
- No tan rápido como SGD puro

**Cuándo usarlo:** La opción recomendada para la mayoría de casos (es el estándar actual).

### 9.2 Tabla Comparativa

```
+-------------------+-------------+---------+------------------+
| Característica    | Batch GD    | SGD     | Mini-Batch GD    |
+-------------------+-------------+---------+------------------+
| Muestras/iter     | Todas (m)   | 1       | n (32-512)       |
+-------------------+-------------+---------+------------------+
| Velocidad         | Lenta       | Rápida  | Media-Rápida     |
+-------------------+-------------+---------+------------------+
| Convergencia      | Suave       | Ruidosa | Relativamente    |
|                   |             |         | suave            |
+-------------------+-------------+---------+------------------+
| Uso de memoria    | Alto        | Bajo    | Medio            |
+-------------------+-------------+---------+------------------+
| Actualizaciones   | Pocas       | Muchas  | Moderadas        |
| por época         |             |         |                  |
+-------------------+-------------+---------+------------------+
| Ideal para        | Datasets    | Datasets| Mayoría de       |
|                   | pequeños    | enormes | casos            |
+-------------------+-------------+---------+------------------+
```

### 9.3 Visualización Conceptual de la Convergencia

Imaginemos cómo cada método "desciende" hacia el mínimo:

**Batch GD:**
```
Inicio → ↓ → ↓ → ↓ → ↓ → Mínimo
         (Camino directo y suave)
```

**SGD:**
```
Inicio → ↓↗↓ → ↙↓↗ → ↓↘↓ → ≈ Mínimo
         (Camino zigzagueante, oscilante)
```

**Mini-Batch GD:**
```
Inicio → ↓ → ↓↘ → ↓ → ↙↓ → Mínimo
         (Camino con ligeras oscilaciones)
```

### 9.4 Tamaños de Batch Comunes

Valores típicos para Mini-Batch GD:

- **32**: Bueno para modelos pequeños, convergencia más ruidosa
- **64**: Balance común, buen punto de partida
- **128**: Estándar para muchos problemas
- **256**: Para datasets grandes
- **512**: Para datasets muy grandes con GPUs potentes

**Regla general:** Potencias de 2 (32, 64, 128...) por eficiencia computacional.

### 9.5 Implementación en Scikit-learn

Scikit-learn permite elegir el optimizador:

```python
# Batch GD (solver predeterminado)
model_batch = LogisticRegression(solver='lbfgs')

# SGD
from sklearn.linear_model import SGDClassifier
model_sgd = SGDClassifier(loss='log_loss')

# Mini-Batch es el comportamiento interno de SGDClassifier
# controlado por el parámetro max_iter y batch size interno
```

### 9.6 Recomendaciones Prácticas

**Para principiantes:**
- Comienza con **Mini-Batch GD** (es el estándar moderno)
- Usa batch size de 64 o 128
- Deja que Scikit-learn maneje los detalles

**Para datasets pequeños (< 10,000):**
- Usa **Batch GD** (solver='lbfgs' en sklearn)
- Converge rápidamente y no necesitas optimización

**Para datasets grandes (> 100,000):**
- Usa **Mini-Batch GD** con SGDClassifier
- Ajusta el learning rate si es necesario

**Nota importante:** En Redes Neuronales profundas, Mini-Batch GD es prácticamente el único método utilizado.

---

Hemos explorado las tres variantes principales del descenso del gradiente. Cada una tiene sus ventajas y casos de uso específicos. En la práctica, Mini-Batch GD es la opción más versátil y recomendada para la mayoría de situaciones. En la siguiente y última sección, resumiremos todo lo aprendido.

## 10. Conclusiones

Hemos completado un recorrido completo por la Regresión Logística, desde su implementación manual hasta su uso profesional con Scikit-learn.

### 10.1 ¿Qué hemos aprendido?

En este artículo hemos cubierto:

**Fundamentos teóricos:**
- La función sigmoide y cómo convierte valores continuos en probabilidades
- La función de coste (Log Loss) para medir el error del modelo
- El descenso del gradiente como método de optimización

**Implementación práctica:**
- División manual de datos en conjuntos de entrenamiento y prueba
- Implementación completa del algoritmo desde cero con Python
- Entrenamiento del modelo y visualización de la convergencia
- Cálculo manual del Accuracy y matriz de confusión

**Herramientas profesionales:**
- Uso de Scikit-learn para división de datos y modelado
- Comparación entre implementación manual y librería profesional
- Predicción en nuevos datos sin etiquetas

**Extensiones y variantes:**
- Clasificación multiclase con estrategias One-vs-Rest y Multinomial
- Tres variantes del descenso del gradiente: Batch GD, SGD y Mini-Batch GD

### 10.2 Resultados Obtenidos

Nuestro modelo de regresión logística en el dataset Breast Cancer Wisconsin logró:

- **Accuracy en entrenamiento:** ~98%
- **Accuracy en prueba:** ~96%
- **Concordancia manual vs Scikit-learn:** 100%

Estos resultados demuestran que la regresión logística, a pesar de su simplicidad, es muy efectiva para problemas linealmente separables.

### 10.3 ¿Cuándo usar Regresión Logística?

**Situaciones ideales:**
- Clasificación binaria o multiclase con pocas categorías
- Cuando necesitas interpretabilidad (entender qué características importan)
- Como baseline antes de probar modelos más complejos
- Cuando tienes datos linealmente separables
- Problemas donde la velocidad de entrenamiento es importante

**Cuándo considerar otros algoritmos:**
- Relaciones no lineales complejas → Random Forest, Gradient Boosting
- Muchas características con interacciones → Redes Neuronales
- Datos no estructurados (imágenes, texto) → Deep Learning
- Problemas con clases muy desbalanceadas → Ajustar pesos o usar métricas alternativas

### 10.4 Ventajas de la Regresión Logística

- **Simplicidad:** Fácil de implementar y entender
- **Interpretabilidad:** Los pesos indican la importancia de cada característica
- **Eficiencia:** Rápida de entrenar, incluso con muchos datos
- **Probabilidades calibradas:** Ofrece probabilidades útiles para toma de decisiones
- **Poco propensa al overfitting:** Especialmente con regularización

### 10.5 Limitaciones

- **Linealidad:** Asume una relación lineal entre características y log-odds
- **Características independientes:** Funciona mejor cuando las características no están altamente correlacionadas
- **Outliers:** Puede ser sensible a valores extremos
- **Frontera de decisión simple:** No captura relaciones complejas sin ingeniería de características

### 10.6 Próximos Pasos Recomendados

Para profundizar en Machine Learning, te recomendamos:

**1. Regularización en Regresión Logística:**
- L1 (Lasso) y L2 (Ridge) para prevenir overfitting
- Elastic Net como combinación de ambas

**2. Métricas de evaluación avanzadas:**
- Precision, Recall, F1-Score
- Curvas ROC y AUC
- Matrices de confusión detalladas

**3. Validación cruzada:**
- K-Fold Cross-Validation
- Stratified K-Fold para datos desbalanceados

**4. Otros algoritmos de clasificación:**
- Support Vector Machines (SVM)
- Decision Trees y Random Forest
- Gradient Boosting (XGBoost, LightGBM)
- Redes Neuronales

**5. Ingeniería de características:**
- Transformaciones polinomiales
- Interacciones entre características
- Feature scaling y normalización

### 10.7 Recursos Adicionales

**Documentación oficial:**
- Scikit-learn: https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
- NumPy: https://numpy.org/doc/stable/
- Pandas: https://pandas.pydata.org/docs/

**Datasets para practicar:**
- UCI Machine Learning Repository
- Kaggle Datasets
- Scikit-learn built-in datasets

**Cursos recomendados:**
- Andrew Ng - Machine Learning (Coursera)
- Fast.ai - Practical Deep Learning
- Google - Machine Learning Crash Course

### 10.8 Reflexión Final

La Regresión Logística es mucho más que un algoritmo simple: es una herramienta fundamental que te permite entender los principios del Machine Learning. Al implementarla desde cero, has desarrollado una comprensión profunda de:

- Cómo los modelos aprenden de los datos
- El papel de las funciones de coste y la optimización
- La importancia de evaluar en datos no vistos
- El balance entre simplicidad e interpretabilidad

Esta base sólida te preparará para abordar algoritmos más complejos con confianza, sabiendo que muchos de ellos comparten estos mismos principios fundamentales.

---

**¡Felicidades por completar este tutorial!** Ahora tienes las herramientas para aplicar Regresión Logística a tus propios problemas de clasificación.

**¿Te ha resultado útil este artículo?** Si tienes preguntas o sugerencias, no dudes en dejar un comentario. ¡Feliz aprendizaje!

---

**Autor:** [Tu nombre]  
**Fecha:** Diciembre 2025  
**GitHub:** [Tu repositorio con el código completo]  
**Kaggle:** [Tu perfil de Kaggle]